In [3]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.SaltRemover import SaltRemover
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import xgboost as xgb
import pickle

# --- File paths ---
train_file = "./dw_data/Opt1_acidic_tr.csv"
test_file  = "./dw_data/Opt1_acidic_tst.csv"

# --- Data Loading ---
train_df = pd.read_csv(train_file)
test_df  = pd.read_csv(test_file)

smiles_col = 'OriginalSmiles'
target = 'pKa'

# Initialize salt remover
saltRemover = SaltRemover(defnFilename='./Salts.txt')

# --- Define fingerprint parameter grid ---
fp_radii = [1, 2, 3, 4]
fp_sizes = [1024, 2048, 4096, 8192, 16384]

# --- Define model hyperparameter grid for XGBoost ---
param_grid = {
    'n_estimators': [300],
    'max_depth': [3, 6],
    'learning_rate': [0.1, 0.2],
    'subsample': [0.8],
    'colsample_bytree': [0.8, 1.0],
    'reg_lambda': [1, 5]
}

best_overall_score = -np.inf
best_model = None
best_fp_radius = None
best_fp_size = None
best_model_params = None
best_metrics = None

# Loop over each fingerprint configuration
for radius in fp_radii:
    for size in fp_sizes:
        print(f"Evaluating fingerprint with radius {radius} and size {size}...")
        
        # Process training data: convert SMILES, remove salts, and generate fingerprints
        rdkit_mols_train = train_df[smiles_col].astype(str).apply(Chem.MolFromSmiles)
        rdkit_mols_train = rdkit_mols_train.apply(saltRemover.StripMol)
        fingerprints_train = rdkit_mols_train.apply(
            lambda mol: AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=size).ToBitString()
        )
        X_train = np.vstack([np.fromstring(fp, 'u1') - ord('0') for fp in fingerprints_train]).astype(float)
        y_train = train_df[target].values
        
        # Process testing data with the same fingerprint parameters
        rdkit_mols_test = test_df[smiles_col].astype(str).apply(Chem.MolFromSmiles)
        rdkit_mols_test = rdkit_mols_test.apply(saltRemover.StripMol)
        fingerprints_test = rdkit_mols_test.apply(
            lambda mol: AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=size).ToBitString()
        )
        X_test = np.vstack([np.fromstring(fp, 'u1') - ord('0') for fp in fingerprints_test]).astype(float)
        y_test = test_df[target].values
        
        # Define and run GridSearchCV for the current fingerprint settings
        xgbr = xgb.XGBRegressor(random_state=42, objective='reg:squarederror')
        grid_search = GridSearchCV(
            estimator=xgbr,
            param_grid=param_grid,
            scoring='r2',
            cv=5,
            n_jobs=5,
            verbose=0
        )
        grid_search.fit(X_train, y_train)
        current_best_model = grid_search.best_estimator_
        
        # Evaluate the model on the test set
        y_pred_test = current_best_model.predict(X_test)
        current_test_r2 = r2_score(y_test, y_pred_test)
        print(f"Test R^2: {current_test_r2:.4f}")
        
        # Update best model if current performance is superior
        if current_test_r2 > best_overall_score:
            best_overall_score = current_test_r2
            best_model = current_best_model
            best_fp_radius = radius
            best_fp_size = size
            best_model_params = grid_search.best_params_
            
            # Also compute train metrics for reporting
            y_pred_train = current_best_model.predict(X_train)
            best_metrics = {
                'Train R^2': r2_score(y_train, y_pred_train),
                'Test R^2': current_test_r2,
                'Test MAE': mean_absolute_error(y_test, y_pred_test),
                'Test MSE': mean_squared_error(y_test, y_pred_test),
                'Test RMSE': np.sqrt(mean_squared_error(y_test, y_pred_test))
            }
            print("New best configuration found!")
        print("-----------------------------------------------------")

print("Best fingerprint parameters and model hyperparameters:")
print(f"Fingerprint radius: {best_fp_radius}")
print(f"Fingerprint size: {best_fp_size}")
print("Model Hyperparameters:", best_model_params)
print("Performance Metrics:", best_metrics)

# Save the best model and fingerprint parameters
with open('best_xgbr_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

with open('fingerprint_params.pkl', 'wb') as f:
    pickle.dump({'FP_radius': best_fp_radius, 'FP_size': best_fp_size}, f)

print("Best model and fingerprint parameters saved to 'best_xgbr_model.pkl' and 'fingerprint_params.pkl'.")


Evaluating fingerprint with radius 1 and size 1024...


C:\Users\Fahad\AppData\Local\Temp\ipykernel_42352\1718603424.py:57: DeprecationWarning: The binary mode of fromstring is deprecated, as it behaves surprisingly on unicode inputs. Use frombuffer instead
  X_train = np.vstack([np.fromstring(fp, 'u1') - ord('0') for fp in fingerprints_train]).astype(float)
C:\Users\Fahad\AppData\Local\Temp\ipykernel_42352\1718603424.py:66: DeprecationWarning: The binary mode of fromstring is deprecated, as it behaves surprisingly on unicode inputs. Use frombuffer instead
  X_test = np.vstack([np.fromstring(fp, 'u1') - ord('0') for fp in fingerprints_test]).astype(float)


Test R^2: 0.7806
New best configuration found!
-----------------------------------------------------
Evaluating fingerprint with radius 1 and size 2048...


C:\Users\Fahad\AppData\Local\Temp\ipykernel_42352\1718603424.py:57: DeprecationWarning: The binary mode of fromstring is deprecated, as it behaves surprisingly on unicode inputs. Use frombuffer instead
  X_train = np.vstack([np.fromstring(fp, 'u1') - ord('0') for fp in fingerprints_train]).astype(float)
C:\Users\Fahad\AppData\Local\Temp\ipykernel_42352\1718603424.py:66: DeprecationWarning: The binary mode of fromstring is deprecated, as it behaves surprisingly on unicode inputs. Use frombuffer instead
  X_test = np.vstack([np.fromstring(fp, 'u1') - ord('0') for fp in fingerprints_test]).astype(float)


Test R^2: 0.8021
New best configuration found!
-----------------------------------------------------
Evaluating fingerprint with radius 1 and size 4096...


...

Test R^2: 0.7460
-----------------------------------------------------
Evaluating fingerprint with radius 4 and size 8192...


C:\Users\Fahad\AppData\Local\Temp\ipykernel_42352\1718603424.py:57: DeprecationWarning: The binary mode of fromstring is deprecated, as it behaves surprisingly on unicode inputs. Use frombuffer instead
  X_train = np.vstack([np.fromstring(fp, 'u1') - ord('0') for fp in fingerprints_train]).astype(float)
C:\Users\Fahad\AppData\Local\Temp\ipykernel_42352\1718603424.py:66: DeprecationWarning: The binary mode of fromstring is deprecated, as it behaves surprisingly on unicode inputs. Use frombuffer instead
  X_test = np.vstack([np.fromstring(fp, 'u1') - ord('0') for fp in fingerprints_test]).astype(float)


Test R^2: 0.7817
-----------------------------------------------------
Evaluating fingerprint with radius 4 and size 16384...


C:\Users\Fahad\AppData\Local\Temp\ipykernel_42352\1718603424.py:57: DeprecationWarning: The binary mode of fromstring is deprecated, as it behaves surprisingly on unicode inputs. Use frombuffer instead
  X_train = np.vstack([np.fromstring(fp, 'u1') - ord('0') for fp in fingerprints_train]).astype(float)
C:\Users\Fahad\AppData\Local\Temp\ipykernel_42352\1718603424.py:66: DeprecationWarning: The binary mode of fromstring is deprecated, as it behaves surprisingly on unicode inputs. Use frombuffer instead
  X_test = np.vstack([np.fromstring(fp, 'u1') - ord('0') for fp in fingerprints_test]).astype(float)


Test R^2: 0.7984
-----------------------------------------------------
Best fingerprint parameters and model hyperparameters:
Fingerprint radius: 1
Fingerprint size: 16384
Model Hyperparameters: {'colsample_bytree': 0.8, 'learning_rate': 0.2, 'max_depth': 6, 'n_estimators': 300, 'reg_lambda': 1, 'subsample': 0.8}
Performance Metrics: {'Train R^2': 0.9725897415088424, 'Test R^2': 0.8133593594900329, 'Test MAE': 0.9056424044000135, 'Test MSE': 2.135269824926312, 'Test RMSE': 1.4612562488921346}
Best model and fingerprint parameters saved to 'best_xgbr_model.pkl' and 'fingerprint_params.pkl'.


In [16]:
import pickle
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.SaltRemover import SaltRemover

# --- Load the saved model and fingerprint parameters ---
with open('best_xgbr_model.pkl', 'rb') as f:
    best_model = pickle.load(f)

with open('fingerprint_params.pkl', 'rb') as f:
    fp_params = pickle.load(f)
FP_radius = fp_params['FP_radius']
FP_size = fp_params['FP_size']

# Initialize the salt remover
saltRemover = SaltRemover(defnFilename='./Salts.txt')

# --- Get user input for the compound identifier and its type ---
compound_input = input("Enter compound identifier (SMILES or IUPAC name): ")
input_type = input("Is the identifier a SMILES string? (y/n): ")

mol = None

if input_type.strip().lower() in ['y', 'yes']:
    # Assume input is a SMILES string
    mol = Chem.MolFromSmiles(compound_input)
    if mol is None:
        print("Error: The provided SMILES string could not be parsed.")
else:
    # Try to convert the IUPAC name to a molecule
    try:
        # Attempt using RDKit's MolFromIUPACName (if available)
        mol = Chem.MolFromIUPACName(compound_input)
    except AttributeError:
        print("Chem.MolFromIUPACName is not available.")
    
    # If still no molecule, attempt fallback using PubChemPy
    if mol is None:
        try:
            import pubchempy as pcp
            compounds = pcp.get_compounds(compound_input, 'name')
            if compounds:
                smiles = compounds[0].isomeric_smiles
                mol = Chem.MolFromSmiles(smiles)
                print(f"Using SMILES from PubChem: {smiles}")
            else:
                print("Error: PubChemPy could not find a compound for the provided name.")
        except ImportError:
            print("Error: PubChemPy is not installed. Install it with 'pip install pubchempy'.")
        except Exception as e:
            print("Error using PubChemPy:", e)

if mol is None:
    print("Error: Could not convert the compound identifier to a molecule. Please check your input.")
else:
    # Remove salts from the molecule
    mol = saltRemover.StripMol(mol)
    
    # Generate fingerprint using the best fingerprint parameters
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=FP_radius, nBits=FP_size).ToBitString()
    X_input = np.fromstring(fp, 'u1') - ord('0')
    X_input = np.array(X_input, dtype=float).reshape(1, -1)
    
    # Predict the pKa using the saved model
    predicted_pka = best_model.predict(X_input)
    print(f"Predicted pKa: {predicted_pka[0]}")


Enter compound identifier (SMILES or IUPAC name):  N1(=C(N=CC=C1)N)
Is the identifier a SMILES string? (y/n):  y


Predicted pKa: 18.07601547241211


C:\Users\Fahad\AppData\Local\Temp\ipykernel_31036\3972216907.py:62: DeprecationWarning: The binary mode of fromstring is deprecated, as it behaves surprisingly on unicode inputs. Use frombuffer instead
  X_input = np.fromstring(fp, 'u1') - ord('0')


In [13]:
!pip install rdkit

  Using cached rdkit-2024.9.6-cp39-cp39-win_amd64.whl.metadata (4.1 kB)
Using cached rdkit-2024.9.6-cp39-cp39-win_amd64.whl (22.5 MB)


ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\Fahad\\anaconda3\\envs\\vs_comparator\\Lib\\site-packages\\rdkit\\rdBase.pyd'
Consider using the `--user` option or check the permissions.

